In [1]:
# Import or install Sionna
import sionna.rt

# Other imports
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import drjit as dr
import mitsuba as mi
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

import tensorflow as tf
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd
import plotly.graph_objects as go

no_preview = False # Toggle to False to use the preview widget

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
    
from sionna.rt import load_scene, PlanarArray, Transmitter, Receiver, ITURadioMaterial,\
    Camera, PathSolver, InteractionType, RadioMapSolver
from sionna.rt.utils import r_hat

2026-02-23 17:17:47.785143: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-23 17:17:47.929307: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771834667.996003  425021 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771834668.013699  425021 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771834668.136865  425021 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [4]:
# ==============================================================================
# 1. XML 파일 경로 보정 및 로드
# ==============================================================================
# 원본 파일 경로 (사용자 환경에 맞게 설정)
xml_path = "/home/js/js/sionna-rt/src/sionna/rt/scenes/kmu_260220/kmu_260220_flat.xml"
scene_dir = os.path.dirname(xml_path)

scene = load_scene(xml_path, merge_shapes=False)

# ==============================================================================
# 2. 재질(Material) 정보 확인
# ==============================================================================
if 'scene' in globals():
    print("\n" + "="*60)
    print(f"{'Object Name':<30} | {'Assigned Material':<20}")
    print("="*60)
    
    # 씬에 있는 모든 객체를 순회하며 할당된 재질 확인
    for name, obj in scene.objects.items():
        # 재질 객체가 있으면 이름 출력, 없으면 None
        mat_name = obj.radio_material.name if obj.radio_material else "None"
        print(f"{name:<30} | {mat_name:<20}")
        
    print("="*60)
    
    # 정의된 모든 재질(Radio Material) 목록 확인
    print("\n[정의된 재질 목록]")
    for mat_name, mat in scene.radio_materials.items():
        print(f" - 이름: {mat_name:<15} (Type: {mat.itu_type}, Thickness: {mat.thickness})")


Object Name                    | Assigned Material   
None_buildings-itu_concrete    | itu_concrete        
None_buildings-itu_metal       | itu_metal           
elm__6                         | itu_concrete        
elm__8                         | itu_concrete        
elm__10                        | itu_concrete        
elm__12                        | itu_concrete        
elm__14                        | itu_concrete        

[정의된 재질 목록]
 - 이름: itu_concrete    (Type: concrete, Thickness: [0.1])
 - 이름: itu_metal       (Type: metal, Thickness: [0.1])


In [ ]:
# ==============================================================================
# 2. 도로 재질 변경 (시각화용)
# ==============================================================================
red_road_mat = ITURadioMaterial(name="red_road_mat_7",
                                itu_type="concrete",
                                thickness=0.2,
                                color=[1.0, 0.0, 0.0])
scene.add(red_road_mat)

road_object_id = "elm__6"
if road_object_id in scene.objects:
    scene.objects[road_object_id].radio_material = red_road_mat
    print(f"[설정] 도로({road_object_id})를 빨간색으로 변경했습니다.")

[설정] 도로(elm__7)를 빨간색으로 변경했습니다.


In [4]:
road_positions = []

print(f"[탐색] Mitsuba Scene 내부에서 '{road_object_id}' 형상을 찾습니다...")

if hasattr(scene, 'mi_scene'):
    mi_scene = scene.mi_scene
    
    # 1. Mitsuba Scene의 모든 Shape를 순회하며 ID 매칭
    target_shape = None
    for s in mi_scene.shapes():
        if s.id() == road_object_id:
            target_shape = s
            break
    
    if target_shape: 
        try:           
            params = mi.traverse(target_shape)
                        
            if 'vertex_positions' in params:
                vertex_buffer = params['vertex_positions']
                
                # NumPy 변환 (1차원 배열: x, y, z, x, y, z ...)
                vertices_flat = np.array(vertex_buffer)
                
                # (N, 3) 형태로 변환 (x, y, z)
                if len(vertices_flat) > 0:
                    vertices = vertices_flat.reshape(-1, 3)
                    road_positions = vertices
                    print(f"  -> 추출된 도로 좌표 수: {len(road_positions)}개")
                else:
                    print("  [경고] 버퍼가 비어있습니다.")
            else:
                print(f"[오류] '{road_object_id}' 객체에 'vertex_positions' 속성이 없습니다.")

        except Exception as e:
            print(f"[오류] Vertex 추출 중 에러 발생: {e}")
    else:
        print(f"[실패] ID가 '{road_object_id}'인 Shape를 mi_scene에서 찾을 수 없습니다.")
else:
    print("[오류] scene 객체에서 'mi_scene' 속성을 찾을 수 없습니다.")

[탐색] Mitsuba Scene 내부에서 'elm__7' 형상을 찾습니다...
  -> 추출된 도로 좌표 수: 226개


In [5]:
x_vals = road_positions[:, 0]
z_vals = road_positions[:, 2] 
indices = list(range(len(road_positions)))

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=x_vals, y=z_vals,
    mode='markers',
    marker=dict(size=5, color='blue'),
    text=indices,
    hovertemplate='<b>Index: %{text}</b><br>X: %{x:.1f}<br>Z: %{y:.1f}<extra></extra>' # 라벨도 Z로 표기
))

fig.update_layout(
    title="도로 점 확인용 지도 (X - Z 평면)",
    xaxis_title="X Axis (East/West)",
    yaxis_title="y Axis (North/South)", # Y축 라벨을 Z축으로 변경
    width=1000, height=800,
    hovermode='closest'
)

fig.show()

In [6]:
# 1. 설정 및 초기화
target_pos = road_positions[0]
print("="*60 + f"\n[설정] 테스트 목표: {target_pos}\n" + "="*60)

# 기존 객체 제거
for name in ['Tx_1', 'Tx_2', 'Tx_3', 'Car_Marker']:
    if name in scene.transmitters: scene.remove(name)
if 'rx_car' in scene.receivers: scene.remove('rx_car')

rx = Receiver(name="rx_car", position=target_pos)
rx.receive_antenna = scene.rx_array
scene.add(rx)
print(" -> 자동차(Rx) 배치 완료.")

cam = Camera(position=target_pos + np.array([0, 100, 100]), look_at=target_pos)
print("[시각화] 3D 뷰어 실행")
scene.preview(show_devices=True, point_picker=True)

[설정] 테스트 목표: [-190.90263    0.3     -138.76404]
 -> 자동차(Rx) 배치 완료.
[시각화] 3D 뷰어 실행


In [7]:
# 1. 설정 및 초기화
target_pos = road_positions[0]
print("="*60 + f"\n[설정] 테스트 목표: {target_pos}\n" + "="*60)

# 2. 장치 배치
scene.tx_array = PlanarArray(num_rows=1, num_cols=1, pattern="iso", polarization="VH")
scene.rx_array = PlanarArray(num_rows=1, num_cols=1, pattern="iso", polarization="VH")

tx_positions = [[-12.176, 25, 84.599]]#, [323.472, 36.869, -204.315], [0.663, 56.367, -181.453]]
tx_names = ["Tx_1"]#, "Tx_2", "Tx_3"]

# 기존 객체 제거
for name in tx_names + ['Car_Marker']:
    if name in scene.transmitters: scene.remove(name)
if 'rx_car' in scene.receivers: scene.remove('rx_car')

for i, pos in enumerate(tx_positions):
    tx = Transmitter(name=tx_names[i], position=pos, power_dbm=43)
    tx.transmit_antenna = scene.tx_array
    tx.look_at(target_pos)
    scene.add(tx)
    print(f" -> {tx_names[i]} 배치 완료.")

rx = Receiver(name="rx_car", position=target_pos)
rx.receive_antenna = scene.rx_array
scene.add(rx)
print(" -> 자동차(Rx) 배치 완료.")

# 3. 시뮬레이션
print("[연산] 경로 계산 시작...")
solver = PathSolver()
paths = solver(scene, max_depth=5, samples_per_src=1000000, diffuse_reflection=True, diffraction=True)

# 4. 결과 검증
a, tau = paths.cir()
if tf.size(a) > 0 and tf.reduce_sum(tf.abs(a)) > 0:
    print(f"\n✅ [성공] 전파 도달 (Amp Sum: {tf.reduce_sum(tf.abs(a)):.2e})")
else:
    print("\n❌ [실패] 전파 미도달 (장애물 또는 거리 문제)")

# 5. 시각화
cam = Camera(position=target_pos + np.array([0, 100, 100]), look_at=target_pos)
print("[시각화] 3D 뷰어 실행")
scene.preview(paths=paths, show_devices=True)

[설정] 테스트 목표: [-190.90263    0.3     -138.76404]
 -> Tx_1 배치 완료.
 -> 자동차(Rx) 배치 완료.
[연산] 경로 계산 시작...

❌ [실패] 전파 미도달 (장애물 또는 거리 문제)
[시각화] 3D 뷰어 실행


2026-02-23 17:00:23.907562: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1928] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 21745 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:67:00.0, compute capability: 8.6
2026-02-23 17:00:23.908914: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1928] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 22190 MB memory:  -> device: 1, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:68:00.0, compute capability: 8.6


In [8]:
# ==============================================================================
# 0. 데이터 준비 & 이동 경로 계산
# ==============================================================================
if 'road_positions' not in globals():
    raise ValueError("❌ 'road_positions' 데이터가 없습니다.")

path_indices = [205, 203, 201, 199, 197, 195, 193, 190, 182, 183, 7, 5]
waypoints = road_positions[path_indices].copy()
waypoints[:, 1] += 1.5

# 한글 주석: 기준 속도(30 km/h -> m/s)
speed_ms = 60.0 / 3.6
delta_t = 0.5

# 한글 주석: 구간 거리/누적 거리 계산
diffs = waypoints[1:] - waypoints[:-1]
segment_dists = np.linalg.norm(diffs, axis=1)
cumulative_dists = np.concatenate(([0], np.cumsum(segment_dists)))
total_distance = cumulative_dists[-1]
total_time = total_distance / speed_ms if speed_ms > 0 else 0.0
time_steps = np.arange(0, total_time + delta_t, delta_t)

def get_state_at_time(t):
    """
    한글 주석: 시각 t에서 차량 위치(pos)와 속도벡터(vel)를 반환
    """
    if len(waypoints) < 2:
        return waypoints[0], np.array([0.0, 0.0, 0.0], dtype=np.float32)

    target_dist = np.clip(speed_ms * t, 0.0, total_distance)
    idx = np.searchsorted(cumulative_dists, target_dist) - 1
    idx = max(0, min(idx, len(waypoints) - 2))

    seg_start = cumulative_dists[idx]
    seg_len = segment_dists[idx]

    if seg_len <= 1e-9:
        return waypoints[idx], np.array([0.0, 0.0, 0.0], dtype=np.float32)

    ratio = (target_dist - seg_start) / seg_len
    pos = waypoints[idx] + ratio * (waypoints[idx+1] - waypoints[idx])

    # 한글 주석: 진행 방향 단위벡터 * 속도크기 = 속도벡터
    direction = (waypoints[idx+1] - waypoints[idx]) / seg_len
    vel = direction * speed_ms
    return pos, vel

# 기존 코드 호환용
def get_pos_at_time(t):
    pos, _ = get_state_at_time(t)
    return pos

# ==============================================================================
# 1. 씬(Scene) 초기화
# ==============================================================================
if hasattr(scene, 'transmitters'):
    for name in list(scene.transmitters.keys()):
        scene.remove(name)
if hasattr(scene, 'receivers'):
    for name in list(scene.receivers.keys()):
        scene.remove(name)

scene.tx_array = PlanarArray(num_rows=1, num_cols=1, pattern="iso", polarization="VH")
scene.rx_array = PlanarArray(num_rows=1, num_cols=1, pattern="iso", polarization="VH")

# 한글 주석: tx_positions, tx_names가 이전 셀에 없으면 기본값 사용
if "tx_positions" not in globals():
    tx_positions = [[-12.176, 25.0, 84.599]]
if "tx_names" not in globals():
    tx_names = ["Tx_1"]

for i, pos in enumerate(tx_positions):
    tx = Transmitter(
        name=tx_names[i],
        position=pos,
        power_dbm=43,
        velocity=[0.0, 0.0, 0.0]  # 한글 주석: 송신기는 정지 가정
    )
    tx.transmit_antenna = scene.tx_array
    tx.look_at([0, 0, 0])
    scene.add(tx)

start_pos, start_vel = get_state_at_time(0.0)
rx = Receiver(
    name="rx_car",
    position=start_pos,
    velocity=start_vel  # 한글 주석: 초기 속도 설정
)
rx.receive_antenna = scene.rx_array
scene.add(rx)

solver = PathSolver()

# ==============================================================================
# 2. 인터랙티브 위젯
# ==============================================================================
output_widget = widgets.Output()

def update_simulation(frame_idx):
    t = float(time_steps[frame_idx])
    current_pos, current_vel = get_state_at_time(t)

    # 한글 주석: 위치/속도 업데이트 (도플러 반영 핵심)
    rx.position = current_pos
    rx.velocity = current_vel
    for name in tx_names:
        scene.transmitters[name].look_at(current_pos)

    paths = solver(
        scene,
        max_depth=3,
        max_num_paths_per_src=10,
        samples_per_src=100000,
        diffuse_reflection=True,
        diffraction=True,
        synthetic_array=True
    )

    with output_widget:
        output_widget.clear_output(wait=True)

        scene.preview(paths=paths, show_devices=True, resolution=[800, 600])

        # 한글 주석: 단일 시각 전력 표시
        a0, _ = paths.cir(out_type="tf", normalize_delays=False)
        p_val = tf.reduce_sum(tf.abs(a0)**2) if tf.size(a0) > 0 else 0.0
        db_val = 10*np.log10(float(p_val) + 1e-30)

        print(f"⏱ Time: {t:.1f}s | 📍 Pos: {current_pos} | 🚗 Vel: {current_vel} m/s | 📶 Power: {db_val:.2f} dB")

slider = widgets.IntSlider(
    value=0, min=0, max=len(time_steps)-1, step=1,
    description='Time Step:', layout=widgets.Layout(width='600px')
)

widgets.interactive_output(update_simulation, {'frame_idx': slider})

print("▼ 슬라이더를 움직여보세요.")
display(slider, output_widget)


▼ 슬라이더를 움직여보세요.


IntSlider(value=0, description='Time Step:', layout=Layout(width='600px'), max=36)

Output()

In [9]:
frame_idx = 0  # 보고 싶은 프레임 인덱스(원하는 값으로 바꿔)
t = time_steps[frame_idx]
current_pos = get_pos_at_time(t)

# 위치/방향 업데이트
rx.position = current_pos
for name in tx_names:
    scene.transmitters[name].look_at(current_pos)

# 경로 계산
paths = solver(scene, max_depth=3, samples_per_src=100000,
               diffuse_reflection=True, diffraction=True,synthetic_array=True)

# CIR 추출
a, tau = paths.cir(out_type="tf", normalize_delays=False)

print("t =", t)
print("a shape:", a.shape)
print("tau shape:", tau.shape)

t = 0.0
a shape: (1, 2, 1, 2, 47, 1)
tau shape: (1, 1, 47)


In [10]:
# 계산 결과 캐시: (frame_idx, tx_idx, rel_delay) -> (t, pos, df)
_pdp_cache = {}

out = widgets.Output()

tx_dropdown = widgets.Dropdown(
    options=[("Tx_1", 0)],#, ("Tx_2", 1), ("Tx_3", 2)],
    value=0,
    description="TX:",
    layout=widgets.Layout(width="200px")
)

frame_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=len(time_steps)-1,
    step=1,
    description="Time:",
    continuous_update=False,
    layout=widgets.Layout(width="600px")
)

# 0부터 시작하는 상대지연으로 볼지 옵션 (원하면 켜기)
rel_delay_chk = widgets.Checkbox(
    value=False,
    description="Relative delay (min τ = 0)",
    indent=False
)

def _compute_mapping_for_frame(frame_idx: int, tx_idx: int, rel_delay: bool):
    key = (frame_idx, tx_idx, rel_delay)
    if key in _pdp_cache:
        return _pdp_cache[key]

    t = float(time_steps[frame_idx])
    pos = get_pos_at_time(t)

    # 위치/방향 업데이트
    rx.position = pos
    for name in tx_names:
        scene.transmitters[name].look_at(pos)

    # 경로 계산
    paths = solver(
        scene,
        max_depth=3,
        samples_per_src=100000,
        diffuse_reflection=True,
        diffraction=True,
        synthetic_array=True
    )

    # CIR 추출 (절대 지연)
    a, tau = paths.cir(out_type="tf", normalize_delays=False)

    # 너 케이스 기준:
    # a: (1, 2, 3, 128, P, 1)
    # tau: (1, 3, P)
    tau_tx = tau[0, tx_idx, :]  

    # PDP: Rx편파(2) + Tx포트(128) 합산 -> (P,)
    pdp = tf.reduce_sum(tf.abs(a[0, :, tx_idx, :, :, 0])**2, axis=[0, 1])  # (P,)

    # ---------------------------
    # padding/가짜 경로 제거
    #   - tau가 음수(-1 같은)면 padding일 가능성이 큼
    # ---------------------------
    valid = tf.math.is_finite(tau_tx) & (tau_tx >= 0)
    idx_valid = tf.where(valid)[:, 0]  # (P_valid,)

    if tf.size(idx_valid) == 0:
        # 유효 경로 없음
        df = pd.DataFrame(columns=["path_idx", "tau_ns", "pdp_db"])
        _pdp_cache[key] = (t, pos, df)
        return _pdp_cache[key]

    tau_v = tf.boolean_mask(tau_tx, valid)   # (P_valid,)
    pdp_v = tf.boolean_mask(pdp, valid)      # (P_valid,)

    # 상대지연 옵션: min τ를 0으로 이동
    if rel_delay:
        tau_v = tau_v - tf.reduce_min(tau_v)

    # 지연 기준 정렬
    order = tf.argsort(tau_v)
    tau_s = tf.gather(tau_v, order).numpy() * 1e9  # ns
    pdp_s = tf.gather(pdp_v, order).numpy()
    pdp_db = 10*np.log10(pdp_s + 1e-30)

    # 정렬된 path_idx (원래 인덱스 기준)
    path_idx_sorted = tf.gather(idx_valid, order).numpy()


    df = pd.DataFrame({
        "path_idx": path_idx_sorted.astype(int),
        "tau_ns": tau_s,
        "pdp_db": pdp_db
        })

    _pdp_cache[key] = (t, pos, df)
    return _pdp_cache[key]

def _update_plot(_=None):
    frame_idx = frame_slider.value
    tx_idx = tx_dropdown.value
    rel_delay = rel_delay_chk.value

    with out:
        out.clear_output(wait=True)

        t, pos, df = _compute_mapping_for_frame(frame_idx, tx_idx, rel_delay)

        if df.empty:
            print(f"t={t:.2f}s | frame={frame_idx} | Tx_{tx_idx+1} : 유효 경로가 없습니다 (완전 차폐/패딩 제거 후 0개).")
            print(f"pos={pos}")
            return

        # 1) 매핑 테이블 출력 (path_idx ↔ 임펄스)
        display(df)

        # 2) CIR(PDP) stem + 라벨(path_idx)
        plt.figure(figsize=(8, 3.6))
        plt.stem(df["tau_ns"].values, df["pdp_db"].values, basefmt=" ")

        # 라벨: 각 임펄스 위에 path_idx
        for x, y, pid in zip(df["tau_ns"].values, df["pdp_db"].values, df["path_idx"].values):
            plt.text(x, y, str(pid), fontsize=9, ha="center", va="bottom")

        plt.xlabel("Delay τ (ns)")
        plt.ylabel("Power (dB)")
        plt.title(f"Tx_{tx_idx+1} PDP at t={t:.2f}s | frame={frame_idx}\npos={pos}")
        plt.grid(True)
        plt.show()

# 이벤트 연결
frame_slider.observe(_update_plot, names="value")
tx_dropdown.observe(_update_plot, names="value")
rel_delay_chk.observe(_update_plot, names="value")

display(widgets.VBox([widgets.HBox([frame_slider, tx_dropdown]), rel_delay_chk]), out)
_update_plot()

Output()